# Stock Cheap Price Checker

This notebook implements a minimal 4-layer stock screener with binary scoring and batch processing.


In [ ]:
import csv
import json
import math
import os
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import requests


# ---------- Layer 1: Universe loader ----------
def get_universe() -> list[str]:
    csv_path = Path("universe.csv")
    if csv_path.exists():
        with csv_path.open("r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            symbols = [row.get("symbol", "").strip().upper() for row in reader]
            symbols = [s for s in symbols if s]
            if symbols:
                return symbols
    return ["AAPL", "MSFT", "GOOGL"]


# ---------- Layer 2: API client ----------
class FinanceApiClient:
    def __init__(
        self,
        api_key: str,
        base_url: str = "https://financialmodelingprep.com/api/v3",
        cache_path: str = ".cache/fmp_cache.json",
        timeout: int = 20,
        retries: int = 3,
    ) -> None:
        self.api_key = api_key
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
        self.retries = retries
        self.session = requests.Session()
        self.cache_path = Path(cache_path)
        self.cache_path.parent.mkdir(parents=True, exist_ok=True)
        self.cache: dict[str, Any] = self._load_cache()

    def _load_cache(self) -> dict[str, Any]:
        if self.cache_path.exists():
            with self.cache_path.open("r", encoding="utf-8") as f:
                return json.load(f)
        return {}

    def _save_cache(self) -> None:
        with self.cache_path.open("w", encoding="utf-8") as f:
            json.dump(self.cache, f)

    def _cache_key(self, endpoint: str, params: dict[str, Any]) -> str:
        return f"{endpoint}?{json.dumps(params, sort_keys=True)}"

    def _get_json(self, endpoint: str, params: dict[str, Any] | None = None) -> Any:
        params = params or {}
        params = dict(params)
        params["apikey"] = self.api_key

        key = self._cache_key(endpoint, params)
        if key in self.cache:
            return self.cache[key]

        url = f"{self.base_url}/{endpoint.lstrip('/')}"
        last_error = None
        for attempt in range(1, self.retries + 1):
            try:
                response = self.session.get(url, params=params, timeout=self.timeout)
                response.raise_for_status()
                data = response.json()
                self.cache[key] = data
                self._save_cache()
                return data
            except requests.RequestException as exc:
                last_error = exc
                if attempt < self.retries:
                    time.sleep(1.25 * attempt)
        raise RuntimeError(f"API request failed for {endpoint}: {last_error}")

    def get_quote(self, symbol: str) -> dict:
        data = self._get_json(f"quote/{symbol}")
        return data[0] if isinstance(data, list) and data else {}

    def get_ratios_ttm(self, symbol: str) -> dict:
        data = self._get_json(f"ratios-ttm/{symbol}")
        return data[0] if isinstance(data, list) and data else {}

    def get_key_metrics(self, symbol: str) -> dict:
        data = self._get_json(f"key-metrics/{symbol}", params={"limit": 5})
        return {"latest": (data[0] if isinstance(data, list) and data else {}), "history": (data if isinstance(data, list) else [])}

    def get_income_statements(self, symbol: str, years: int = 5) -> list[dict]:
        data = self._get_json(f"income-statement/{symbol}", params={"limit": years})
        return data if isinstance(data, list) else []

    def get_cash_flows(self, symbol: str, years: int = 5) -> list[dict]:
        data = self._get_json(f"cash-flow-statement/{symbol}", params={"limit": years})
        return data if isinstance(data, list) else []


# ---------- Layer 3: Metric calculator ----------
def _to_float(value: Any) -> float | None:
    if value is None:
        return None
    try:
        value = float(value)
        if math.isfinite(value):
            return value
    except (TypeError, ValueError):
        return None
    return None


def compute_gross_margin(income_stmt: dict) -> float | None:
    gross_profit = _to_float(income_stmt.get("grossProfit"))
    revenue = _to_float(income_stmt.get("revenue"))
    if not gross_profit or not revenue:
        return None
    return gross_profit / revenue


def compute_fcf_margin(cash_flow: dict, revenue: float | None) -> float | None:
    free_cash_flow = _to_float(cash_flow.get("freeCashFlow"))
    if not free_cash_flow or not revenue:
        return None
    return free_cash_flow / revenue


def cagr(start: float, end: float, periods: int) -> float | None:
    if periods <= 0 or start <= 0 or end <= 0:
        return None
    return (end / start) ** (1 / periods) - 1


def compute_cagr(values: list[float]) -> float | None:
    clean = [v for v in values if v is not None and v > 0]
    if len(clean) < 2:
        return None
    return cagr(clean[0], clean[-1], len(clean) - 1)


def compute_relative_valuation(current: float | None, historical_avg: float | None) -> float | None:
    if current is None or historical_avg is None or historical_avg <= 0:
        return None
    return current / historical_avg


def _mean(values: list[float | None]) -> float | None:
    clean = [v for v in values if v is not None and v > 0]
    if not clean:
        return None
    return sum(clean) / len(clean)


# ---------- Layer 4: Scorer ----------
def score_stock(metrics: dict) -> dict:
    quality_score = 0.0
    valuation_score = 0.0

    if (metrics.get("gross_margin") or 0) >= 0.50:
        quality_score += 0.50
    if (metrics.get("net_margin") or 0) >= 0.15:
        quality_score += 0.15
    if (metrics.get("fcf_margin") or 0) >= 0.15:
        quality_score += 0.15
    if (metrics.get("roic") or 0) >= 0.20:
        quality_score += 0.20
    if (metrics.get("revenue_growth_5y") or 0) >= 0.10:
        quality_score += 0.20
    if (metrics.get("eps_growth_5y") or 0) >= 0.10:
        quality_score += 0.20

    current_pe = metrics.get("current_pe")
    avg_5y_pe = metrics.get("avg_5y_pe")
    current_ev_ebitda = metrics.get("current_ev_ebitda")
    avg_5y_ev_ebitda = metrics.get("avg_5y_ev_ebitda")
    current_mc_fcf = metrics.get("current_mc_fcf")
    avg_5y_mc_fcf = metrics.get("avg_5y_mc_fcf")

    if current_pe is not None and avg_5y_pe is not None and current_pe < avg_5y_pe:
        valuation_score += 0.20
    if current_ev_ebitda is not None and avg_5y_ev_ebitda is not None and current_ev_ebitda < avg_5y_ev_ebitda:
        valuation_score += 0.20
    if current_mc_fcf is not None and avg_5y_mc_fcf is not None and current_mc_fcf < avg_5y_mc_fcf:
        valuation_score += 0.20

    total_score = quality_score + valuation_score
    return {
        "quality_score": quality_score,
        "valuation_score": valuation_score,
        "total_score": total_score,
    }


# ---------- Orchestration ----------
RESULT_COLUMNS = [
    "symbol",
    "price",
    "gross_margin",
    "net_margin",
    "fcf_margin",
    "roic",
    "revenue_growth_5y",
    "eps_growth_5y",
    "current_pe",
    "avg_5y_pe",
    "current_ev_ebitda",
    "avg_5y_ev_ebitda",
    "current_mc_fcf",
    "avg_5y_mc_fcf",
    "quality_score",
    "valuation_score",
    "total_score",
    "screened_at",
]


def _chunks(items: list[str], n: int) -> list[list[str]]:
    return [items[i : i + n] for i in range(0, len(items), n)]


def _append_rows_to_csv(rows: list[dict], output_csv: str) -> None:
    out = Path(output_csv)
    write_header = not out.exists()
    with out.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=RESULT_COLUMNS)
        if write_header:
            writer.writeheader()
        writer.writerows(rows)


def _safe_get(d: dict, *keys: str) -> float | None:
    for k in keys:
        if k in d:
            v = _to_float(d.get(k))
            if v is not None:
                return v
    return None


def process_symbol(symbol: str, client: FinanceApiClient) -> dict:
    quote = client.get_quote(symbol)
    ratios_ttm = client.get_ratios_ttm(symbol)
    key_metrics_data = client.get_key_metrics(symbol)
    key_metrics_latest = key_metrics_data.get("latest", {})
    key_metrics_history = key_metrics_data.get("history", [])
    income_stmts = client.get_income_statements(symbol, years=5)
    cash_flows = client.get_cash_flows(symbol, years=5)

    latest_income = income_stmts[0] if income_stmts else {}
    latest_cash_flow = cash_flows[0] if cash_flows else {}

    gross_margin = compute_gross_margin(latest_income)
    revenue = _to_float(latest_income.get("revenue"))
    net_income = _to_float(latest_income.get("netIncome"))
    net_margin = (net_income / revenue) if net_income and revenue else None
    fcf_margin = compute_fcf_margin(latest_cash_flow, revenue)
    roic = _safe_get(key_metrics_latest, "roic", "returnOnInvestedCapital")

    rev_series = [
        _to_float(stmt.get("revenue"))
        for stmt in reversed(income_stmts)
    ]
    eps_series = [
        _to_float(stmt.get("eps"))
        for stmt in reversed(income_stmts)
    ]

    revenue_growth_5y = compute_cagr(rev_series)
    eps_growth_5y = compute_cagr(eps_series)

    current_pe = _safe_get(ratios_ttm, "peRatioTTM", "peRatio", "priceEarningsRatioTTM")
    current_ev_ebitda = _safe_get(ratios_ttm, "enterpriseValueMultipleTTM", "evToEbitda")
    current_mc_fcf = _safe_get(ratios_ttm, "priceToFreeCashFlowsRatioTTM", "priceToFreeCashFlowsRatio")

    hist_pe = [_safe_get(row, "peRatio") for row in key_metrics_history]
    hist_ev_ebitda = [_safe_get(row, "evToEbitda", "enterpriseValueOverEBITDA") for row in key_metrics_history]
    hist_mc_fcf = [_safe_get(row, "priceToFreeCashFlowsRatio") for row in key_metrics_history]

    avg_5y_pe = _mean(hist_pe)
    avg_5y_ev_ebitda = _mean(hist_ev_ebitda)
    avg_5y_mc_fcf = _mean(hist_mc_fcf)

    metrics = {
        "gross_margin": gross_margin,
        "net_margin": net_margin,
        "fcf_margin": fcf_margin,
        "roic": roic,
        "revenue_growth_5y": revenue_growth_5y,
        "eps_growth_5y": eps_growth_5y,
        "current_pe": current_pe,
        "avg_5y_pe": avg_5y_pe,
        "current_ev_ebitda": current_ev_ebitda,
        "avg_5y_ev_ebitda": avg_5y_ev_ebitda,
        "current_mc_fcf": current_mc_fcf,
        "avg_5y_mc_fcf": avg_5y_mc_fcf,
    }

    scores = score_stock(metrics)

    # Relative valuation ratios are computed for interpretation but not required in output columns.
    _ = compute_relative_valuation(current_pe, avg_5y_pe)
    _ = compute_relative_valuation(current_ev_ebitda, avg_5y_ev_ebitda)
    _ = compute_relative_valuation(current_mc_fcf, avg_5y_mc_fcf)

    row = {
        "symbol": symbol,
        "price": _to_float(quote.get("price")),
        "gross_margin": gross_margin,
        "net_margin": net_margin,
        "fcf_margin": fcf_margin,
        "roic": roic,
        "revenue_growth_5y": revenue_growth_5y,
        "eps_growth_5y": eps_growth_5y,
        "current_pe": current_pe,
        "avg_5y_pe": avg_5y_pe,
        "current_ev_ebitda": current_ev_ebitda,
        "avg_5y_ev_ebitda": avg_5y_ev_ebitda,
        "current_mc_fcf": current_mc_fcf,
        "avg_5y_mc_fcf": avg_5y_mc_fcf,
        "quality_score": scores["quality_score"],
        "valuation_score": scores["valuation_score"],
        "total_score": scores["total_score"],
        "screened_at": datetime.now(timezone.utc).isoformat(),
    }
    return row


def run_screening(
    output_csv: str = "stock_screen_results.csv",
    batch_size: int = 25,
    sleep_seconds: float = 1.5,
) -> list[dict]:
    api_key = os.getenv("FMP_API_KEY", "")
    if not api_key:
        raise ValueError("Set FMP_API_KEY in your environment before running.")

    symbols = get_universe()
    client = FinanceApiClient(api_key=api_key)

    completed: set[str] = set()
    out = Path(output_csv)
    if out.exists():
        with out.open("r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            completed = {row.get("symbol", "") for row in reader if row.get("symbol")}

    all_new_rows: list[dict] = []
    pending = [s for s in symbols if s not in completed]

    for batch in _chunks(pending, batch_size):
        rows: list[dict] = []
        for symbol in batch:
            try:
                rows.append(process_symbol(symbol, client))
            except Exception as exc:
                print(f"Failed: {symbol} -> {exc}")

        if rows:
            _append_rows_to_csv(rows, output_csv)
            all_new_rows.extend(rows)

        time.sleep(sleep_seconds)

    all_rows = []
    if out.exists():
        with out.open("r", newline="", encoding="utf-8") as f:
            all_rows = list(csv.DictReader(f))
    return all_rows


# Example run:
# results = run_screening(output_csv="stock_screen_results.csv", batch_size=25, sleep_seconds=1.5)
# results[:3]
